# Every LLM Has a Latent World Model Inside
## Full Pipeline: D0 + D1 + D2

**Runtime**: Seleziona **GPU** da `Runtime > Change runtime type > T4 GPU` (o meglio, A100 se disponibile).

**Tempo stimato**:
- D0 + D1 (3 geometrie ciascuno): ~15-20 min su T4
- D2 encoding: ~10-15 min (LM scoring è la parte lenta)
- D2 training: ~20-30 min

Puoi runnare le sezioni indipendentemente — D0/D1 non richiedono D2 e viceversa.

## 0. Setup

In [ ]:
# Clona il repo
!git clone https://github.com/PiGrieco/Every_LLM_has_a_Latent_World_Model_inside.git
%cd Every_LLM_has_a_Latent_World_Model_inside

In [ ]:
# Installa dipendenze
!pip install -q torch numpy scipy transformers sentence-transformers \
    datasets faiss-cpu pyyaml tqdm matplotlib seaborn umap-learn scikit-learn

In [ ]:
# Verifica GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 1. D0 — Synthetic Time-Reversal (H1 + H2)

Il test fondamentale: il metric Lorentziano distingue traiettorie forward da reversed?

Runna tutte e 3 le geometrie (Lorentzian, Riemannian, Euclidean) per confronto.

In [ ]:
!python -m scripts.train --config configs/d0_synthetic.yaml --geometry lorentzian

In [ ]:
!python -m scripts.train --config configs/d0_synthetic.yaml --geometry riemannian

In [ ]:
!python -m scripts.train --config configs/d0_synthetic.yaml --geometry euclidean

In [ ]:
# Confronta risultati D0
import json, os

print("=" * 70)
print("D0 RESULTS — TIME-REVERSAL EXPERIMENT")
print("=" * 70)
print(f"{'Geometry':<15} {'M1 (timelike)':<18} {'M2 (action gap)':<20} {'M2 (logprob gap)'}")
print("-" * 70)

for geo in ["lorentzian", "riemannian", "euclidean"]:
    path = f"./outputs/d0/d0_results_{geo}.json"
    if os.path.exists(path):
        r = json.load(open(path))
        m1 = r.get("m1_timelike_rate", "N/A")
        m2a = r.get("m2_action_gap", "N/A")
        m2l = r.get("m2_logprob_gap", "N/A")
        print(f"{geo:<15} {m1:<18.4f} {m2a:<20.4f} {m2l:.4f}")
    else:
        print(f"{geo:<15} (not run)")

In [ ]:
# Mostra i plot D0
from IPython.display import Image, display
import glob

for img in sorted(glob.glob("./outputs/d0/*.png")):
    print(f"\n--- {os.path.basename(img)} ---")
    display(Image(filename=img, width=600))

---
## 2. D1 — Synthetic Branching (H3)

Le continuazioni alternative sono separate da intervalli space-like?
Solo Lorentzian può ottenere BOTH time-like within-branch AND space-like cross-branch.

In [ ]:
!python -m scripts.train --config configs/d1_branching.yaml --geometry lorentzian

In [ ]:
!python -m scripts.train --config configs/d1_branching.yaml --geometry riemannian

In [ ]:
!python -m scripts.train --config configs/d1_branching.yaml --geometry euclidean

In [ ]:
# Confronta risultati D1
print("=" * 90)
print("D1 RESULTS — BRANCHING EXPERIMENT")
print("=" * 90)
print(f"{'Geometry':<15} {'M1':<10} {'M3':<10} {'M3p within':<14} {'M3p cross':<14} {'M3p joint':<14}")
print("-" * 77)

for geo in ["lorentzian", "riemannian", "euclidean"]:
    path = f"./outputs/d1/d1_results_{geo}.json"
    if os.path.exists(path):
        r = json.load(open(path))
        m1 = r.get("m1_timelike_rate", 0)
        m3 = r.get("m3_branching_separation", 0)
        m3w = r.get("m3p_within_timelike", 0)
        m3c = r.get("m3p_cross_spacelike", 0)
        m3j = r.get("m3p_joint", 0)
        print(f"{geo:<15} {m1:<10.4f} {m3:<10.4f} {m3w:<14.4f} {m3c:<14.4f} {m3j:<14.4f}")
    else:
        print(f"{geo:<15} (not run)")

In [ ]:
# Plot D1
for img in sorted(glob.glob("./outputs/d1/*.png")):
    print(f"\n--- {os.path.basename(img)} ---")
    display(Image(filename=img, width=600))

---
## 3. D2 — WikiText-103 Real Text (la tesi vera)

### Step 1: Pre-encode il corpus
Questo step calcola:
- Embeddings dei paragrafi (sentence-transformer)
- Log-prob LM per ogni transizione (gpt2-medium)

Il risultato viene cachato in `./cache/` — basta runnarlo una volta.

In [ ]:
# Pre-encode WikiText-103 (embeddings + LM scores)
# ~10-15 min su T4 GPU
!python -m scripts.encode_corpus --config configs/d2_wikitext.yaml

In [ ]:
# Verifica che la cache esista
import os
cache_dir = "./cache"
if os.path.exists(cache_dir):
    for f in sorted(os.listdir(cache_dir)):
        size = os.path.getsize(os.path.join(cache_dir, f))
        print(f"  {f:<45} {size / 1e6:.1f} MB")
else:
    print("Cache non trovata — runna la cella precedente.")

### Step 2: Train D2 (Lorentzian)

In [ ]:
# Train D2 con geometria Lorentziana
!python -m scripts.train --config configs/d2_wikitext.yaml

### Step 3 (opzionale): Train D2 baselines

In [ ]:
# Riemannian baseline
!python -m scripts.train --config configs/d2_wikitext.yaml --geometry riemannian

In [ ]:
# Euclidean baseline
!python -m scripts.train --config configs/d2_wikitext.yaml --geometry euclidean

In [ ]:
# Confronta risultati D2
print("=" * 80)
print("D2 RESULTS — WIKITEXT-103 REAL TEXT")
print("=" * 80)
print(f"{'Geometry':<15} {'M1':<10} {'M4 Jaccard':<14} {'M4 Prec':<12} {'M4 Recall':<12} {'M5 NLL':<10}")
print("-" * 73)

for geo in ["lorentzian", "riemannian", "euclidean"]:
    path = f"./outputs/d2/d2_results_{geo}.json"
    if os.path.exists(path):
        r = json.load(open(path))
        m1 = r.get("m1", r.get("m1_timelike_rate", 0))
        m4j = r.get("m4_jaccard", 0)
        m4p = r.get("m4_precision", 0)
        m4r = r.get("m4_recall", 0)
        m5 = r.get("m5", r.get("m5_nll", 0))
        print(f"{geo:<15} {m1:<10.4f} {m4j:<14.4f} {m4p:<12.4f} {m4r:<12.4f} {m5:<10.4f}")
    else:
        print(f"{geo:<15} (not run)")

In [ ]:
# Plot D2
for img in sorted(glob.glob("./outputs/d2/*.png")):
    print(f"\n--- {os.path.basename(img)} ---")
    display(Image(filename=img, width=600))

---
## 4. Alternativa rapida: run_baselines (D0 + D1 in un colpo)

Se vuoi runnare D0 e D1 con tutte e 3 le geometrie in automatico:

In [ ]:
# ALTERNATIVA: runna tutto D0 + D1 in automatico
# (salta questa cella se hai già runnato le sezioni sopra)
# !python -m scripts.run_baselines

---
## 5. Download risultati

Scarica output, checkpoint e plot sul tuo computer.

In [ ]:
# Comprimi tutto per download
!tar czf results.tar.gz outputs/ checkpoints/ cache/
print(f"results.tar.gz: {os.path.getsize('results.tar.gz') / 1e6:.1f} MB")

# Su Colab, scarica con:
try:
    from google.colab import files
    files.download('results.tar.gz')
except ImportError:
    print("Non su Colab — scarica manualmente results.tar.gz")